# LeetCode 2094 — Finding 3-Digit Even Numbers bằng Python

**Link đề:** https://leetcode.com/problems/finding-3-digit-even-numbers/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Recursion, Sorting, Enumeration

```python
class Solution:
    def findEvenNumbers(self, digits: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu ba điều kiện: ghép **3 phần tử khác vị trí**, **không có số 0 đứng đầu**, **số chẵn**; kết quả **không trùng** và **tăng dần**.
2. Đi từ **thử mọi bộ 3 chỉ số** (`O(n³)`) → `itertools.permutations` → **đảo chiều suy nghĩ**: thử mọi đáp án 100..998 rồi kiểm tra bằng **bảng đếm chữ số** (`O(1)` theo n sau khi đếm).
3. Nắm cú pháp: `set`, `sorted`, `//` và `%` để tách chữ số, `itertools.permutations`, `Counter`, mảng đếm `[0]*10`, `range(100, 1000, 2)`.
4. Trace, chứng minh, Time/Space, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và cách hiểu đúng

Cho mảng `digits`, mỗi phần tử là một chữ số (có thể trùng). Tìm **mọi số nguyên khác nhau** thỏa:

1. Ghép từ **ba phần tử** của `digits` (ba **vị trí khác nhau**, thứ tự tùy ý).
2. **Không có số 0 đứng đầu** (tức là số có đúng 3 chữ số, từ 100 trở lên).
3. Là **số chẵn** (chữ số hàng đơn vị là 0, 2, 4, 6, 8).

Trả về danh sách **tăng dần, không trùng**.

| `digits` | Kết quả |
|---|---|
| `[2,1,3,0]` | `[102,120,130,132,210,230,302,310,312,320]` |
| `[2,2,8,8,2]` | `[222,228,282,288,822,828,882]` |
| `[3,7,5]` | `[]` |

**Constraints:** `3 <= len(digits) <= 100`, `0 <= digits[i] <= 9`.

Ý quan trọng: **mỗi chữ số dùng tối đa số lần nó xuất hiện**. Với `[2,2,8,8,2]`: được dùng `8` hai lần (288), nhưng `888` thì **không** được vì chỉ có hai số 8.

In [1]:
examples = [
    ([2, 1, 3, 0], [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]),
    ([2, 2, 8, 8, 2], [222, 228, 282, 288, 822, 828, 882]),
    ([3, 7, 5], []),
]
for d, exp in examples:
    print(d, "->", exp)

[2, 1, 3, 0] -> [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]
[2, 2, 8, 8, 2] -> [222, 228, 282, 288, 822, 828, 882]
[3, 7, 5] -> []


## 2. Nền tảng Python cần dùng

### 2.1. Ghép ba chữ số thành một số: `a*100 + b*10 + c`

Chữ số hàng trăm `a`, hàng chục `b`, hàng đơn vị `c`. Ví dụ `a=1, b=3, c=2` → `132`.

In [2]:
a, b_, c = 1, 3, 2
print(a * 100 + b_ * 10 + c)   # 132
print(0 * 100 + 1 * 10 + 2)    # 12 -> có số 0 đứng đầu, KHÔNG hợp lệ

132
12


### 2.2. Tách chữ số bằng `//` và `%`

- `x % 10` → chữ số cuối.
- `x // 10` → bỏ chữ số cuối.
- `divmod(x, 10)` trả cùng lúc `(x // 10, x % 10)`.

Với số 3 chữ số `x`: hàng trăm `x // 100`, hàng chục `x // 10 % 10`, hàng đơn vị `x % 10`.

In [3]:
x = 528
print(x // 100, x // 10 % 10, x % 10)   # 5 2 8
print(divmod(528, 10))                  # (52, 8)
print("528 chẵn?", x % 2 == 0)

5 2 8
(52, 8)
528 chẵn? True


### 2.3. `set` loại trùng, `sorted` sắp xếp

Nhiều bộ chỉ số khác nhau có thể tạo ra **cùng một số** (ví dụ hai số 2 khác vị trí). `set` giữ mỗi giá trị một lần; `sorted(set)` trả list tăng dần.

In [4]:
s = set()
for v in [222, 228, 222, 282, 228]:
    s.add(v)
print(s)
print(sorted(s))

{282, 228, 222}
[222, 228, 282]


### 2.4. `itertools.permutations(iterable, r)`

Sinh mọi **bộ có thứ tự** gồm `r` phần tử lấy ở **các vị trí khác nhau**. Phần tử trùng giá trị ở hai vị trí khác nhau vẫn được coi là khác → có bộ lặp giá trị.

In [5]:
from itertools import permutations
print(list(permutations([1, 2, 3], 2)))
print(list(permutations([2, 2, 8], 3)))   # có bộ lặp vì hai số 2 khác vị trí

[(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
[(2, 2, 8), (2, 8, 2), (2, 2, 8), (2, 8, 2), (8, 2, 2), (8, 2, 2)]


### 2.5. Bảng đếm chữ số: `[0] * 10` hoặc `Counter`

`cnt[d]` = số lần chữ số `d` xuất hiện. Vì chữ số chỉ từ 0 đến 9, một list 10 ô là đủ.

In [6]:
from collections import Counter
digits = [2, 2, 8, 8, 2]
cnt = [0] * 10
for d in digits:
    cnt[d] += 1
print("cnt  =", cnt)                 # cnt[2]=3, cnt[8]=2
print("Counter =", Counter(digits))

cnt  = [0, 0, 3, 0, 0, 0, 0, 0, 2, 0]
Counter = Counter({2: 3, 8: 2})


### 2.6. `range(start, stop, step)` với bước nhảy

`range(100, 1000, 2)` sinh 100, 102, ..., 998 — **mọi số chẵn có 3 chữ số**. Chỉ có 450 số!

In [7]:
evens = range(100, 1000, 2)
print(len(evens), evens[0], evens[1], evens[-1])

450 100 102 998


## 3. Cách 1 — Trực giác: thử mọi bộ ba chỉ số khác nhau

Ba vòng lặp lồng `i, j, k` (đôi một khác nhau) chọn chữ số hàng trăm, chục, đơn vị. Kiểm tra: `digits[i] != 0` và `digits[k]` chẵn. Bỏ vào `set` rồi sắp xếp.

In [8]:
def find_even_brute(digits: list[int]) -> list[int]:
    n = len(digits)
    found = set()
    for i in range(n):
        if digits[i] == 0:              # không cho số 0 đứng đầu
            continue
        for j in range(n):
            if j == i:
                continue
            for k in range(n):
                if k == i or k == j:
                    continue
                if digits[k] % 2 == 0:  # số chẵn
                    found.add(digits[i] * 100 + digits[j] * 10 + digits[k])
    return sorted(found)

for d, exp in examples:
    print(find_even_brute(d) == exp, find_even_brute(d))

True [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]
True [222, 228, 282, 288, 822, 828, 882]
True []


### Cách 1b — Cùng ý tưởng, dùng `permutations`

In [9]:
def find_even_perm(digits):
    found = set()
    for a, b_, c in permutations(digits, 3):
        if a != 0 and c % 2 == 0:
            found.add(a * 100 + b_ * 10 + c)
    return sorted(found)

for d, exp in examples:
    print(find_even_perm(d) == exp)

True
True
True


## 4. Nút thắt của cách thử mọi bộ chỉ số

Với `n = 100`: số bộ `(i, j, k)` đôi một khác nhau là `100·99·98 ≈ 970.000`. Chạy được, nhưng:

- Phần lớn bộ tạo ra **số trùng** (vì chỉ có 10 chữ số khác nhau!) → lãng phí.
- Nếu `n` lên `10^4` thì `10^12` bộ → không thể.

**Quan sát then chốt:** dù `digits` dài bao nhiêu, đáp án chỉ có thể là **số chẵn từ 100 đến 998** — tối đa **450 ứng viên**. Đây chính là gợi ý của đề.

In [10]:
for n in [10, 100, 1000]:
    print(f"n={n:>5}: số bộ chỉ số = {n*(n-1)*(n-2):,}  | số ứng viên đáp án luôn = 450")

n=   10: số bộ chỉ số = 720  | số ứng viên đáp án luôn = 450
n=  100: số bộ chỉ số = 970,200  | số ứng viên đáp án luôn = 450
n= 1000: số bộ chỉ số = 997,002,000  | số ứng viên đáp án luôn = 450


## 5. Bước đột phá — Đảo chiều: thử từng đáp án, kiểm tra có ghép được không

Thay vì "từ chữ số → sinh số", ta làm "từ số → kiểm tra chữ số":

1. Đếm `cnt[d]` cho `digits` (một lần, `O(n)`).
2. Với mỗi `x` trong `range(100, 1000, 2)`: tách 3 chữ số, đếm nhu cầu `need[d]`.
3. `x` hợp lệ ⇔ với mọi chữ số `d`, `need[d] <= cnt[d]`.

Ưu điểm phụ: duyệt `x` **tăng dần** nên kết quả **tự sắp xếp** và **không trùng** — không cần `set` hay `sorted`.

Ví dụ: `digits=[2,2,8,8,2]` có `cnt[8]=2`. Ứng viên `888` cần `need[8]=3 > 2` → loại. Ứng viên `288` cần `need[2]=1, need[8]=2` → hợp lệ.

In [11]:
cnt = Counter([2, 2, 8, 8, 2])
for x in [888, 288, 228, 220]:
    need = Counter([x // 100, x // 10 % 10, x % 10])
    ok = all(need[d] <= cnt[d] for d in need)
    print(f"x={x}: need={dict(need)} -> {'hợp lệ' if ok else 'loại'}")

x=888: need={8: 3} -> loại
x=288: need={2: 1, 8: 2} -> hợp lệ
x=228: need={2: 2, 8: 1} -> hợp lệ
x=220: need={2: 2, 0: 1} -> loại


## 6. Lời giải chính — mổ xẻ từng đoạn rồi ghép lại

### 6.1. Đoạn 1: đếm chữ số của input

In [12]:
def count_digits(digits):
    cnt = [0] * 10
    for d in digits:
        cnt[d] += 1
    return cnt

print(count_digits([2, 1, 3, 0]))

[1, 1, 1, 1, 0, 0, 0, 0, 0, 0]


### 6.2. Đoạn 2: hàm kiểm tra một ứng viên `x`

Ta copy nhu cầu vào một list 10 ô (`need`), rồi so sánh từng ô. Để gọn, có thể trừ dần từ bản sao của `cnt`: nếu ô nào âm → không đủ.

In [13]:
def can_form(x, cnt):
    need = [0] * 10
    for d in (x // 100, x // 10 % 10, x % 10):
        need[d] += 1
    return all(need[d] <= cnt[d] for d in range(10))

cnt = count_digits([2, 2, 8, 8, 2])
print(can_form(288, cnt), can_form(888, cnt), can_form(100, cnt))

True False False


### 6.3. Ghép: duyệt mọi số chẵn 3 chữ số

`range(100, 1000, 2)` tự đảm bảo hai điều kiện "không có 0 đứng đầu" và "chẵn" → chỉ còn kiểm tra chữ số.

In [14]:
def find_even_numbers(digits: list[int]) -> list[int]:
    cnt = count_digits(digits)
    return [x for x in range(100, 1000, 2) if can_form(x, cnt)]

for d, exp in examples:
    print(find_even_numbers(d) == exp, find_even_numbers(d))

True [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]
True [222, 228, 282, 288, 822, 828, 882]
True []


## 7. TRACE từng bước

### 7.1. Bảng trace cho `digits = [2, 1, 3, 0]` (một vài ứng viên)

`cnt = [1,1,1,1,0,0,0,0,0,0]` (mỗi chữ số 0,1,2,3 có một lần).

| x | chữ số | need ≠ 0 | So với cnt | Kết luận |
|:---:|---|---|---|---|
| 100 | 1,0,0 | need[0]=2, need[1]=1 | need[0]=2 > cnt[0]=1 | loại |
| 102 | 1,0,2 | mỗi ô 1 | đều ≤ 1 | **giữ** |
| 104 | 1,0,4 | need[4]=1 | cnt[4]=0 | loại |
| 110 | 1,1,0 | need[1]=2 | > 1 | loại |
| 120 | 1,2,0 | mỗi ô 1 | đều ≤ 1 | **giữ** |

In [15]:
def trace_find_even(digits, show=lambda x: True):
    cnt = count_digits(digits)
    print("cnt =", cnt)
    result = []
    for x in range(100, 1000, 2):
        need = [0] * 10
        for d in (x // 100, x // 10 % 10, x % 10):
            need[d] += 1
        bad = [d for d in range(10) if need[d] > cnt[d]]
        ok = not bad
        if ok:
            result.append(x)
        if show(x):
            reason = "giữ" if ok else f"thiếu chữ số {bad}"
            print(f"  x={x}: {reason}")
    print("Kết quả:", result)
    return result

trace_find_even([2, 1, 3, 0], show=lambda x: x <= 140)

cnt = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
  x=100: thiếu chữ số [0]
  x=102: giữ
  x=104: thiếu chữ số [4]
  x=106: thiếu chữ số [6]
  x=108: thiếu chữ số [8]
  x=110: thiếu chữ số [1]
  x=112: thiếu chữ số [1]
  x=114: thiếu chữ số [1, 4]
  x=116: thiếu chữ số [1, 6]
  x=118: thiếu chữ số [1, 8]
  x=120: giữ
  x=122: thiếu chữ số [2]
  x=124: thiếu chữ số [4]
  x=126: thiếu chữ số [6]
  x=128: thiếu chữ số [8]
  x=130: giữ
  x=132: giữ
  x=134: thiếu chữ số [4]
  x=136: thiếu chữ số [6]
  x=138: thiếu chữ số [8]
  x=140: thiếu chữ số [4]
Kết quả: [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]


[102, 120, 130, 132, 210, 230, 302, 310, 312, 320]

### 7.2. Nhánh đặc biệt: chữ số lặp (dùng được tối đa số lần có)

In [16]:
trace_find_even([2, 2, 8, 8, 2], show=lambda x: x in (222, 228, 282, 288, 822, 828, 882, 888, 280))

cnt = [0, 0, 3, 0, 0, 0, 0, 0, 2, 0]
  x=222: giữ
  x=228: giữ
  x=280: thiếu chữ số [0]
  x=282: giữ
  x=288: giữ
  x=822: giữ
  x=828: giữ
  x=882: giữ
  x=888: thiếu chữ số [8]
Kết quả: [222, 228, 282, 288, 822, 828, 882]


[222, 228, 282, 288, 822, 828, 882]

### 7.3. Trường hợp không có chữ số chẵn → rỗng

In [17]:
trace_find_even([3, 7, 5], show=lambda x: x in (370, 572, 753 - 1))

cnt = [0, 0, 0, 1, 0, 1, 0, 1, 0, 0]
  x=370: thiếu chữ số [0]
  x=572: thiếu chữ số [2]
  x=752: thiếu chữ số [2]
Kết quả: []


[]

## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — Ba vòng lặp chỉ số + set

In [18]:
class Solution:
    def findEvenNumbers(self, digits: list[int]) -> list[int]:
        n = len(digits)
        found = set()
        for i in range(n):
            if digits[i] == 0:
                continue
            for j in range(n):
                if j == i:
                    continue
                for k in range(n):
                    if k != i and k != j and digits[k] % 2 == 0:
                        found.add(digits[i] * 100 + digits[j] * 10 + digits[k])
        return sorted(found)

print([Solution().findEvenNumbers(d) == e for d, e in examples])

[True, True, True]


### Cách B — `itertools.permutations` + set

In [19]:
from itertools import permutations

class Solution:
    def findEvenNumbers(self, digits: list[int]) -> list[int]:
        return sorted({a * 100 + b * 10 + c
                       for a, b, c in permutations(digits, 3)
                       if a != 0 and c % 2 == 0})

print([Solution().findEvenNumbers(d) == e for d, e in examples])

[True, True, True]


### Cách C — Thử 450 ứng viên, kiểm tra bằng `Counter` (lời giải chính)

`Counter(str(x))` đếm ký tự của số; so với `Counter(map(str, digits))`. Phép `need - cnt` trên Counter chỉ giữ các ô **dương** → rỗng nghĩa là đủ chữ số.

In [20]:
from collections import Counter

print(Counter("288") - Counter("22882"))   # Counter() -> đủ
print(Counter("888") - Counter("22882"))   # Counter({'8': 1}) -> thiếu một số 8

class Solution:
    def findEvenNumbers(self, digits: list[int]) -> list[int]:
        have = Counter(map(str, digits))
        return [x for x in range(100, 1000, 2) if not (Counter(str(x)) - have)]

print([Solution().findEvenNumbers(d) == e for d, e in examples])

Counter()
Counter({'8': 1})
[True, True, True]


### Cách D — Ba vòng lặp trên **giá trị chữ số** (0..9) với mảng đếm

Không duyệt chỉ số, mà duyệt giá trị: `a` từ 1..9, `b` từ 0..9, `c` từ 0,2,4,6,8. Mỗi lần chọn thì **giảm** `cnt`, chọn xong **trả lại** (giống backtracking). Thứ tự duyệt là tăng dần nên kết quả tự sắp xếp, không trùng.

In [21]:
class Solution:
    def findEvenNumbers(self, digits: list[int]) -> list[int]:
        cnt = [0] * 10
        for d in digits:
            cnt[d] += 1
        res = []
        for a in range(1, 10):
            if cnt[a] == 0:
                continue
            cnt[a] -= 1
            for b in range(10):
                if cnt[b] == 0:
                    continue
                cnt[b] -= 1
                for c in range(0, 10, 2):
                    if cnt[c] > 0:
                        res.append(a * 100 + b * 10 + c)
                cnt[b] += 1      # trả lại
            cnt[a] += 1          # trả lại
        return res

print([Solution().findEvenNumbers(d) == e for d, e in examples])

[True, True, True]


## 9. Vì sao thuật toán đúng?

**Cách C (thử ứng viên):**
- *Không bỏ sót:* mọi số hợp lệ là số chẵn có 3 chữ số (không 0 đứng đầu ⇒ ≥ 100; ≤ 999; chẵn) ⇒ nằm trong `range(100, 1000, 2)`. Nếu nó ghép được từ 3 vị trí khác nhau thì mỗi chữ số `d` được dùng không quá `cnt[d]` lần ⇒ vượt qua kiểm tra.
- *Không nhận nhầm:* nếu `need[d] <= cnt[d]` với mọi `d`, ta chọn được `need[d]` vị trí khác nhau mang chữ số `d` cho từng `d` ⇒ có 3 vị trí khác nhau ghép thành `x`.
- *Không trùng, đã sắp xếp:* mỗi `x` được xét đúng một lần theo thứ tự tăng.

**Cách D:** invariant — tại vòng trong cùng, `cnt` = số chữ số **còn lại** sau khi đã lấy `a` và `b`; nhờ trả lại sau vòng lặp, mỗi nhánh bắt đầu với bảng đếm đúng. Thứ tự (a, b, c) tăng dần theo từ điển ⇔ `100a + 10b + c` tăng dần.

## 10. Phân tích Time / Space Complexity

Gọi `n = len(digits)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| A. 3 vòng chỉ số | `O(n³)` (~10^6 khi n=100) | `O(R)` set kết quả | đơn giản nhưng lãng phí |
| B. permutations | `O(n³)` | `O(R)` | gọn, cùng độ phức tạp |
| C. 450 ứng viên | `O(n + 450·10)` = `O(n)` | `O(1)` (10 ô) | lời giải chính |
| D. Giá trị 0..9 | `O(n + 9·10·5)` = `O(n)` | `O(1)` | nhanh nhất, không cần sort |

`R ≤ 450` là số kết quả. Phần sau khi đếm của C và D là **hằng số** (không phụ thuộc `n`).

In [22]:
import random, time
_r = random.Random(1)
big = [_r.randint(0, 9) for _ in range(100)]
for name, f in [("brute", find_even_brute), ("perm", find_even_perm), ("candidates", find_even_numbers)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:10}: {len(r)} số, {dt*1000:.1f} ms")

brute     : 450 số, 112.1 ms


perm      : 450 số, 100.3 ms
candidates: 450 số, 0.4 ms


## 11. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [23]:
def sol_D(digits):
    cnt = [0] * 10
    for d in digits: cnt[d] += 1
    res = []
    for a in range(1, 10):
        if not cnt[a]: continue
        cnt[a] -= 1
        for b_ in range(10):
            if not cnt[b_]: continue
            cnt[b_] -= 1
            res += [a * 100 + b_ * 10 + c for c in range(0, 10, 2) if cnt[c]]
            cnt[b_] += 1
        cnt[a] += 1
    return res

SOLS = [find_even_brute, find_even_perm, find_even_numbers, sol_D]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:30} PASS ({len(actual)} số)")

tests = examples + [
    ([0, 0, 0], []),              # chỉ có 0 -> mọi số có 0 đứng đầu
    ([1, 0, 0], [100]),           # đúng một số
    ([2, 2, 2], [222]),
    ([8, 8, 8, 8], [888]),
    ([1, 1, 1], []),              # không có chữ số chẵn
    ([0, 2, 0], [200]),
]
for i, (d, exp) in enumerate(tests):
    for f in SOLS:
        check(f"test {i} {f.__name__}", f(d), exp)

test 0 find_even_brute         PASS (10 số)
test 0 find_even_perm          PASS (10 số)
test 0 find_even_numbers       PASS (10 số)
test 0 sol_D                   PASS (10 số)
test 1 find_even_brute         PASS (7 số)
test 1 find_even_perm          PASS (7 số)
test 1 find_even_numbers       PASS (7 số)
test 1 sol_D                   PASS (7 số)
test 2 find_even_brute         PASS (0 số)
test 2 find_even_perm          PASS (0 số)
test 2 find_even_numbers       PASS (0 số)
test 2 sol_D                   PASS (0 số)
test 3 find_even_brute         PASS (0 số)
test 3 find_even_perm          PASS (0 số)
test 3 find_even_numbers       PASS (0 số)
test 3 sol_D                   PASS (0 số)
test 4 find_even_brute         PASS (1 số)
test 4 find_even_perm          PASS (1 số)
test 4 find_even_numbers       PASS (1 số)
test 4 sol_D                   PASS (1 số)
test 5 find_even_brute         PASS (1 số)
test 5 find_even_perm          PASS (1 số)
test 5 find_even_numbers       PASS (1 số)
test 5 

In [24]:
rng = random.Random(2094)
for _ in range(200):
    d = [rng.randint(0, 9) for _ in range(rng.randint(3, 9))]
    outs = [f(d) for f in SOLS]
    assert all(o == outs[0] for o in outs), d
print("200 test ngẫu nhiên: 4 cách khớp nhau")
# Tính chất: mọi số đều chẵn, >= 100, tăng dần nghiêm ngặt
r = find_even_numbers(list(range(10)) * 3)
assert all(x % 2 == 0 and 100 <= x <= 998 for x in r) and all(r[i] < r[i + 1] for i in range(len(r) - 1))
print("Khi có đủ mọi chữ số 3 lần:", len(r), "số (= 450)")

200 test ngẫu nhiên: 4 cách khớp nhau


Khi có đủ mọi chữ số 3 lần: 450 số (= 450)


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: Cho phép dùng lại cùng một vị trí (quên `i != j != k`)

In [25]:
def wrong_reuse(digits):
    found = set()
    for a in digits:
        for b_ in digits:
            for c in digits:
                if a != 0 and c % 2 == 0:
                    found.add(a * 100 + b_ * 10 + c)
    return sorted(found)

print("Sai :", wrong_reuse([3, 7, 2])[:5], "... (222 không được phép!)")
print("Đúng:", find_even_numbers([3, 7, 2]))

Sai : [222, 232, 272, 322, 332] ... (222 không được phép!)
Đúng: [372, 732]


**Sửa:** kiểm tra chỉ số khác nhau, hoặc dùng `permutations`, hoặc so sánh với bảng đếm.

### Lỗi 2: Kiểm tra "chữ số có mặt" thay vì "đủ số lần"

In [26]:
def wrong_presence(digits):
    have = set(digits)
    return [x for x in range(100, 1000, 2)
            if {x // 100, x // 10 % 10, x % 10} <= have]   # SAI: bỏ qua số lần

print("Sai có 888?", 888 in wrong_presence([2, 2, 8, 8, 2]), "| Đúng có 888?", 888 in find_even_numbers([2, 2, 8, 8, 2]))

Sai có 888? True | Đúng có 888? False


### Lỗi 3: Quên loại số 0 đứng đầu

Nếu cho `a = 0`, ta tạo ra số 2 chữ số như `012 = 12` → sai đề.

In [27]:
def wrong_leading_zero(digits):
    return sorted({a * 100 + b_ * 10 + c for a, b_, c in permutations(digits, 3) if c % 2 == 0})

print("Sai :", wrong_leading_zero([2, 1, 3, 0])[:4], "| Đúng:", find_even_numbers([2, 1, 3, 0])[:4])

Sai : [12, 32, 102, 120] | Đúng: [102, 120, 130, 132]


### Lỗi 4: Quên khử trùng hoặc quên sắp xếp ở cách sinh

In [28]:
raw = [a * 100 + b_ * 10 + c for a, b_, c in permutations([2, 2, 8, 8, 2], 3) if a and c % 2 == 0]
print("Số phần tử khi chưa khử trùng:", len(raw), "| sau set:", len(set(raw)))
print("Chưa sắp xếp:", raw[:6])

Số phần tử khi chưa khử trùng: 60 | sau set: 7
Chưa sắp xếp: [228, 228, 222, 282, 288, 282]


## 13. Mẹo nhận dạng pattern

- **Không gian đáp án nhỏ hơn không gian input** → *đảo chiều*: liệt kê đáp án, kiểm tra từng cái (enumerate the answer).
- "Dùng mỗi phần tử tối đa số lần nó có" → **bảng đếm** và so sánh `need <= have` (như 383 Ransom Note, 1160 Find Words That Can Be Formed by Characters).
- Phần tử là chữ số / chữ cái → bảng đếm cỡ 10 / 26 thay vì dict.
- Sinh tổ hợp có thứ tự nhỏ → `itertools.permutations` / `product` / backtracking với "lấy rồi trả lại".

## 14. Biến thể

### 14.1. Số có `k` chữ số chia hết cho 5

In [29]:
def find_k_digit_div5(digits, k):
    have = Counter(map(str, digits))
    lo, hi = 10 ** (k - 1), 10 ** k
    return [x for x in range(lo, hi) if x % 5 == 0 and not (Counter(str(x)) - have)]

print(find_k_digit_div5([1, 5, 0, 2], 2))

[10, 15, 20, 25, 50]


### 14.2. Chỉ cần **đếm** số lượng số chẵn 3 chữ số tạo được

In [30]:
def count_even_numbers(digits):
    return len(find_even_numbers(digits))

print(count_even_numbers([2, 1, 3, 0]), count_even_numbers([2, 2, 8, 8, 2]))

10 7


### 14.3. Số lớn nhất chẵn 3 chữ số tạo được (hoặc -1)

Duyệt ngược `range(998, 99, -2)` và dừng ở số đầu tiên hợp lệ.

In [31]:
def largest_even(digits):
    cnt = count_digits(digits)
    for x in range(998, 99, -2):
        if can_form(x, cnt):
            return x
    return -1

print(largest_even([2, 1, 3, 0]), largest_even([3, 7, 5]))

320 -1


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Đáp án ở **mục 17**.

### Bài 1 — Tách 3 chữ số của một số
`split3(528) -> (5, 2, 8)`

In [32]:
def split3(x):
    # TODO: viết code ở đây
    pass

# assert split3(528) == (5, 2, 8)

### Bài 2 — Kiểm tra ghép được không
`can_build(x, digits)` trả True nếu số `x` (3 chữ số) ghép được từ `digits` (mỗi phần tử dùng tối đa 1 lần).

In [33]:
def can_build(x, digits):
    # TODO: viết code ở đây
    pass

# assert can_build(288, [2, 2, 8, 8, 2]) is True
# assert can_build(888, [2, 2, 8, 8, 2]) is False

### Bài 3 — Tự viết lại lời giải #2094 theo cách thử ứng viên

In [34]:
def find_even_ex(digits):
    # TODO: viết code ở đây
    pass

# assert find_even_ex([2, 1, 3, 0]) == [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]

### Bài 4 — Biến thể: số 3 chữ số **lẻ** và **đối xứng** (palindrome) ghép được

In [35]:
def odd_palindromes(digits):
    # TODO: viết code ở đây
    pass

# assert odd_palindromes([1, 2, 1, 3]) == [121, 131]

### Bài 5 — Tự tạo test làm lộ **Lỗi 2** (chỉ kiểm tra có mặt)

In [36]:
my_tests = [
    # TODO: các digits mà wrong_presence(d) != find_even_numbers(d)
]
# for d in my_tests:
#     assert wrong_presence(d) != find_even_numbers(d)

### Bài 6 — Mini challenge: số chẵn **4 chữ số** nhỏ nhất ghép được (hoặc -1)

In [37]:
def smallest_even_4(digits):
    # TODO: viết code ở đây
    pass

# assert smallest_even_4([3, 0, 1, 2]) == 1032
# assert smallest_even_4([1, 3, 5, 7]) == -1

## 16. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `//` và `%` | hàng trăm = `x // 100` | `(x//100, x//10%10, x%10)` |
| 2 | so sánh bảng đếm | `Counter` của x và digits | `not (Counter(str(x)) - Counter(map(str, digits)))` |
| 3 | liệt kê đáp án | `range(100, 1000, 2)` | dùng bài 2 làm bộ lọc |
| 4 | đổi bước nhảy và thêm điều kiện | `range(101, 1000, 2)` | `x // 100 == x % 10` |
| 5 | lỗi bỏ qua số lần | cần chữ số dùng 2 lần mà chỉ có 1 | `[1, 0, 2]` → `100` không ghép được |
| 6 | giống bài 3 với 4 chữ số | `range(1000, 10000, 2)` | trả số đầu tiên hợp lệ |

## 17. Đáp án tham khảo

In [38]:
def split3(x):
    return (x // 100, x // 10 % 10, x % 10)

def can_build(x, digits):
    return not (Counter(str(x)) - Counter(map(str, digits)))

def find_even_ex(digits):
    return [x for x in range(100, 1000, 2) if can_build(x, digits)]

def odd_palindromes(digits):
    return [x for x in range(101, 1000, 2) if x // 100 == x % 10 and can_build(x, digits)]

my_tests = [[1, 0, 2], [2, 0, 4], [1, 2, 3, 4]]

def smallest_even_4(digits):
    for x in range(1000, 10000, 2):
        if can_build(x, digits):
            return x
    return -1

assert split3(528) == (5, 2, 8)
assert can_build(288, [2, 2, 8, 8, 2]) is True and can_build(888, [2, 2, 8, 8, 2]) is False
assert find_even_ex([2, 1, 3, 0]) == [102, 120, 130, 132, 210, 230, 302, 310, 312, 320]
assert odd_palindromes([1, 2, 1, 3]) == [121, 131]
for d in my_tests:
    assert wrong_presence(d) != find_even_numbers(d)
assert smallest_even_4([3, 0, 1, 2]) == 1032 and smallest_even_4([1, 3, 5, 7]) == -1
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


**Giải thích bằng lời:**
- Bài 1: tách chữ số bằng chia nguyên và chia lấy dư.
- Bài 2: `Counter(x) - Counter(digits)` rỗng ⇔ mọi chữ số của `x` đủ số lần.
- Bài 3: đảo chiều — thử 450 ứng viên.
- Bài 4: palindrome 3 chữ số ⇔ hàng trăm = hàng đơn vị; lẻ ⇒ bước 2 từ 101.
- Bài 5: cần một số đòi một chữ số 2 lần trong khi chỉ có 1 lần (ví dụ `100`, `200`, `112`...).
- Bài 6: duyệt tăng dần nên số đầu tiên hợp lệ là nhỏ nhất.

## 18. Tổng kết — template + checklist

**Template "liệt kê đáp án + bảng đếm":**
```python
have = [0] * 10
for d in digits: have[d] += 1
for x in candidates_in_order:
    if need(x) <= have (từng ô): giữ x
```

**Checklist trước khi submit:**
- [ ] Không số 0 đứng đầu (≥ 100).
- [ ] Chẵn (chữ số cuối chẵn).
- [ ] Mỗi chữ số dùng không quá số lần có.
- [ ] Không trùng, tăng dần.
- [ ] Đúng tên method `findEvenNumbers`.
- [ ] Ví dụ `[3,7,5]` trả `[]`.